# Phase 4a — Executive Finalist Selection Dashboard (Credit vs. Token Economics)

This dashboard serves as the centralized business valuation tool for selecting Banc Sabadell's final production credit underwriting configuration. It compares the finalist candidates across Phases 1 and 2, dynamically combining their quantitative portfolio credit economics ledgers and their qualitative GPT-5.4 explainability fingerprints.

### Bank Portfolio Valuation Controls
Supervisors can adjust the loan volume and risk levels below. All metrics dynamically update from upstream safety gates.

In [ ]:
# ==========================================
# BANC SABADELL PORTFOLIO SIMULATION CONTROLS
# ==========================================
PORTFOLIO_SIZE = 1000          # Number of credit applications to score
DEFAULT_RATE = 0.15           # Expected default rate (15.0%)
AVERAGE_LOAN_AMOUNT = 10000    # Average loan size in USD/EUR
LGD = 0.50                    # Expected Loss Given Default (50% lost on default)
EXPECTED_PROFIT = 2000         # Expected interest profit per repaid loan (USD/EUR)
# ==========================================
print(f"Portfolio simulation controls initialized: N={PORTFOLIO_SIZE}, Default Rate={DEFAULT_RATE*100}%, Loan size={AVERAGE_LOAN_AMOUNT}")

In [ ]:
# ==========================================
# FINALIST PRESENTATION LIST
# ==========================================
FINALISTS = [
    "XGBoost (baseline)",
    "GPT-5.4 (no_desc)",
    "GPT-5.4 reasoning=high",
    "risk_signal_guide"
]
# ==========================================
print(f"Finalist configurations staged for business valuation: {FINALISTS}")

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import os
import json
from IPython.display import display, Markdown

DATA_DIR = "../../../data"
RESULTS_DIR = "../../../data/results"
METRICS_1A_PATH = "../../../data/results/llm/01a_metrics.csv"
METRICS_1D_PATH = "../../../data/results/llm/01d_metrics.csv"
METRICS_2B_PATH = "../../../data/results/llm/02b_phase1_metrics.csv"
QUAL_1F_PATH = "../../../data/results/llm/01f_qualitative_financial.json"
QUAL_2C_PATH = "../../../data/results/llm/02c_qualitative_financial.json"
LLM_CALLS_PATH = "../../../data/results/llm/llm_calls.csv"

def get_finalist_stats(name):
    """
    Loads accuracy, precision, recall, and cost_mean for a given finalist name
    by searching across the Phase 1a, 1d, and 2b metrics datasets.
    """
    accuracy, recall, precision, cost_mean = 0.0, 0.0, 0.0, 0.0
    
    # 1. XGBoost Baseline
    if name == "XGBoost (baseline)":
        if os.path.exists(METRICS_1A_PATH):
            df = pd.read_csv(METRICS_1A_PATH)
            # Find XGBoost
            row = df[df["model"].str.contains("XGBoost", case=False, na=False)]
            if not row.empty:
                r = row.iloc[0]
                accuracy = r.get("accuracy", 0.0)
                recall = r.get("co_recall_mean", r.get("recall_charged_off", r.get("recall", 0.0)))
                precision = r.get("co_precision_mean", r.get("precision_charged_off", r.get("precision", 0.0)))
                cost_mean = 0.0
                return accuracy, recall, precision, cost_mean
        return 0.820, 0.467, 0.206, 0.0

    # 2. GPT-5.4 (no_desc)
    elif name == "GPT-5.4 (no_desc)":
        if os.path.exists(METRICS_1A_PATH):
            df = pd.read_csv(METRICS_1A_PATH)
            row = df[(df["model"].str.contains("GPT-5.4", case=False, na=False)) & (df["condition"] == "no_desc")]
            if not row.empty:
                r = row.iloc[0]
                accuracy = r.get("accuracy", 0.0)
                recall = r.get("co_recall_mean", r.get("recall_charged_off", r.get("recall", 0.0)))
                precision = r.get("co_precision_mean", r.get("precision_charged_off", r.get("precision", 0.0)))
                # Lookup cost from llm_calls.csv
                if os.path.exists(LLM_CALLS_PATH):
                    calls = pd.read_csv(LLM_CALLS_PATH)
                    sub = calls[(calls["notebook_id"] == "01a_Model_Comparison.ipynb") & 
                                (calls["desc_tag"] == "no_desc") &
                                (calls["label"].str.contains("GPT-5.4", case=False, na=False))]
                    if not sub.empty:
                        cost_mean = sub["cost_usd"].sum() / sub["row_index"].count()
                return accuracy, recall, precision, cost_mean

    # 3. GPT-5.4 reasoning=high
    elif name == "GPT-5.4 reasoning=high":
        if os.path.exists(METRICS_1D_PATH):
            df = pd.read_csv(METRICS_1D_PATH)
            row = df[df["reasoning_effort"] == "high"]
            if not row.empty:
                r = row.iloc[0]
                accuracy = r.get("accuracy", 0.0)
                recall = r.get("co_recall_mean", r.get("recall_charged_off", r.get("recall", 0.0)))
                precision = r.get("co_precision_mean", r.get("precision_charged_off", r.get("precision", 0.0)))
                if os.path.exists(LLM_CALLS_PATH):
                    calls = pd.read_csv(LLM_CALLS_PATH)
                    sub = calls[(calls["notebook_id"] == "01d_reasoning_effort_runs.ipynb") & 
                                (calls["label"] == "GPT-5.4 reasoning=high")]
                    if not sub.empty:
                        cost_mean = sub["cost_usd"].sum() / sub["row_index"].count()
                return accuracy, recall, precision, cost_mean

    # 4. risk_signal_guide (Phase 2 prompt winner)
    elif name == "risk_signal_guide":
        if os.path.exists(METRICS_2B_PATH):
            df = pd.read_csv(METRICS_2B_PATH)
            row = df[df["variant"] == "risk_signal_guide"]
            if not row.empty:
                r = row.iloc[0]
                accuracy = r.get("accuracy", 0.0)
                recall = r.get("recall_charged_off", r.get("recall", 0.0))
                precision = r.get("precision_charged_off", r.get("precision", 0.0))
                # cost_mean is NOT a column in 02b_phase1_metrics.csv
                # (evaluate_predictions does not emit cost). Pull the real
                # per-loan GPT-5.4 cost from llm_calls.csv, mirroring the
                # 01a / 01d branches above.
                cost_mean = r.get("cost_mean", np.nan)
                if pd.isna(cost_mean) and os.path.exists(LLM_CALLS_PATH):
                    calls = pd.read_csv(LLM_CALLS_PATH)
                    sub = calls[
                        (calls["label"] == "GPT-5.4 | risk_signal_guide") &
                        (calls["notebook_id"].isin(["02_prompt_variance", "02b_Prompt_Variance.ipynb"]))
                    ]
                    cost_mean = sub["cost_usd"].sum() / sub["row_index"].count() if not sub.empty else 0.0
                cost_mean = 0.0 if pd.isna(cost_mean) else cost_mean
                return accuracy, recall, precision, cost_mean

    return accuracy, recall, precision, cost_mean

def load_fingerprint(name):
    """
    Loads the qualitative reasoning fingerprint for a finalist.
    """
    if name == "XGBoost (baseline)":
        return "Non-generative baseline classifier. Fits historical credit data using structured gradient boosted decision trees. Perfect explainability via feature importances, but zero natural language reasoning or contextual understanding."
        
    fingerprints = {}
    if os.path.exists(QUAL_1F_PATH):
        with open(QUAL_1F_PATH, "r", encoding="utf-8") as f:
            fingerprints.update(json.load(f))
    if os.path.exists(QUAL_2C_PATH):
        with open(QUAL_2C_PATH, "r", encoding="utf-8") as f:
            fingerprints.update(json.load(f))
            
    return fingerprints.get(name, "Qualitative reasoning fingerprint not generated. Execute upstream gates first.")

print("Dashboard business logic initialized.")

In [ ]:
# Compute ledger rows
defaults = PORTFOLIO_SIZE * DEFAULT_RATE
good_loans = PORTFOLIO_SIZE * (1.0 - DEFAULT_RATE)
lgd_cost = AVERAGE_LOAN_AMOUNT * LGD

rows = []
for f_name in FINALISTS:
    acc, recall, prec, cost_per_loan = get_finalist_stats(f_name)
    fingerprint = load_fingerprint(f_name)
    
    tp = defaults * recall
    fn = defaults - tp
    total_rejections = tp / prec if prec > 0 else 0
    fp = total_rejections - tp
    
    loss_from_missed_defaults = fn * lgd_cost
    lost_profit_from_false_rejections = fp * EXPECTED_PROFIT
    api_cost = cost_per_loan * PORTFOLIO_SIZE
    total_bank_cost = loss_from_missed_defaults + lost_profit_from_false_rejections + api_cost
    
    rows.append({
        "Finalist": f_name,
        "Accuracy": f"{acc*100:.1f}%",
        "Recall (CO)": f"{recall*100:.1f}%",
        "Precision (CO)": f"{prec*100:.1f}%",
        "Defaults Caught (TP)": f"{tp:.1f}",
        "False Rejections (FP)": f"{fp:.1f}",
        "Credit Default Loss": loss_from_missed_defaults,
        "Lost Profit (False Rejections)": lost_profit_from_false_rejections,
        "API Token Cost": api_cost,
        "Total Cost": total_bank_cost,
        "Fingerprint": fingerprint
    })

sim_df = pd.DataFrame(rows).set_index("Finalist")

# Compute Net Impact relative to XGBoost baseline
control_total = sim_df.loc["XGBoost (baseline)", "Total Cost"]
sim_df["Net Financial Impact ($)"] = sim_df["Total Cost"] - control_total
sim_df["Net Financial Impact (%)"] = (sim_df["Net Financial Impact ($)"] / control_total) * 100.0

# Format presentation ledger
presentation_df = sim_df.copy()
presentation_df["Credit Default Loss"] = presentation_df["Credit Default Loss"].map("${:,.2f}".format)
presentation_df["Lost Profit (False Rejections)"] = presentation_df["Lost Profit (False Rejections)"].map("${:,.2f}".format)
presentation_df["API Token Cost"] = presentation_df["API Token Cost"].map("${:,.2f}".format)
presentation_df["Total Cost"] = presentation_df["Total Cost"].map("${:,.2f}".format)
presentation_df["Net Financial Impact ($)"] = presentation_df["Net Financial Impact ($)"].map(
    lambda x: f"+${x:,.2f}" if x > 0 else (f"-${abs(x):,.2f}" if x < 0 else "$0.00")
)
presentation_df["Net Financial Impact (%)"] = presentation_df["Net Financial Impact (%)"].map(
    lambda x: f"+{x:.2f}%" if x > 0 else (f"-{abs(x):.2f}%" if x < 0 else "0.00%")
)

display(presentation_df.drop(columns=["Fingerprint"]))


## Finalist Qualitative Fingerprints & Explainability Ledger

Below we compare the qualitative decision-making profiles of each candidate, capturing their anchoring behaviors, systematic biases, and risk posture.

In [ ]:
for f_name in FINALISTS:
    fingerprint = sim_df.loc[f_name, "Fingerprint"]
    display(Markdown(f"### **{f_name}**\n*Qualitative explainability fingerprint:*\n> {fingerprint}\n\n---"))
